In [0]:
spark.conf.set(
    "fs.azure.account.key.lcdevadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcstgadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcprodadls.blob.core.windows.net",
    "REDACTED"
)

In [0]:
# ==============================
# Notebook: Parikshit_LC_Borrower_Insights.ipynb
# Author: Parikshit Narang
# Purpose: Lending Club Borrower Analysis (Using preprocessed dataset)
# Features: Borrower segmentation, behavioral analysis, geographic insights, cluster visualization
# ==============================

# ------------------------------
# 1. Import Libraries
# ------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg, sum as Fsum, months_between, current_date
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

import pandas as pd
import plotly.express as px
from pyspark.sql import functions as F

import plotly.io as pio

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import year, month

import plotly.express as px
import plotly.io as pio

import plotly.express as px
import plotly.io as pio


import plotly.offline as pyo

pio.renderers.default = 'databricks' 

In [0]:
# ------------------------------
# 3. Load Preprocessed Dataset
# ------------------------------
processed_path = "wasbs://lcloansdev@lcdevadls.blob.core.windows.net/"
df = spark.read.parquet(processed_path)

# Cast problematic string columns to numeric
numeric_cols = ["revol_util", "dti", "delinq_2yrs", "annual_inc"]
for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double"))

print(f"Loaded preprocessed dataset with {df.count()} rows and {len(df.columns)} columns.")
df.show(5)



In [0]:
# ------------------------------
# 4. Feature Selection for Clustering
# ------------------------------
# We'll use key numeric features and optionally encode categorical features
numeric_features = ["revol_util", "dti", "total_bal_ex_mort", "num_accts_ever_120_pd", "delinq_2yrs", "annual_inc"]
categorical_features = ["home_ownership", "purpose", "grade", "sub_grade", "term", "verification_status"]

# Fill null values in numeric features (if any) with 0 or other suitable value
df = df.fillna({col: 0 for col in numeric_features})

# Fill null values in categorical features with "UNKNOWN" (or another placeholder)
df = df.fillna({col: "UNKNOWN" for col in categorical_features})

# Index + OneHotEncode categorical columns
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="skip") for c in categorical_features]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec") for c in categorical_features]

# Assemble features
assembler = VectorAssembler(
    inputCols=numeric_features + [c+"_vec" for c in categorical_features],
    outputCol="features",
    handleInvalid="skip"  # Skip rows with invalid data
)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Define KMeans clustering
kmeans = KMeans(featuresCol="scaledFeatures", k=5, seed=42, predictionCol="cluster")

# Full pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, kmeans])
model = pipeline.fit(df)
df_clusters = model.transform(df)


In [0]:
# ------------------------------
# 5. K-Means Clustering
# ------------------------------
kmeans = KMeans(featuresCol="scaledFeatures", k=5, seed=42, predictionCol="cluster")

# Full pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, kmeans])
model = pipeline.fit(df)
df_clusters = model.transform(df)

In [0]:
# ------------------------------
# Cluster Profiling Setup
# ------------------------------

# Compute cluster statistics
cluster_stats = df_clusters.groupBy("cluster").agg(
    F.avg("revol_util").alias("avg_revol_util"),
    F.avg("dti").alias("avg_dti"),
    F.avg("total_bal_ex_mort").alias("avg_total_bal_ex_mort"),
    F.avg("num_accts_ever_120_pd").alias("avg_num_120_pd"),
    F.avg("delinq_2yrs").alias("avg_delinq_2yrs"),
    F.avg("annual_inc").alias("avg_annual_inc")
).orderBy("cluster")

# Convert to Pandas
cluster_df = cluster_stats.toPandas()

# Map cluster numbers to names
cluster_names = {
    0: "Stable Borrowers",
    1: "High Risk Borrowers",
    2: "Potentially Over-leveraged",
    3: "Low Debt Borrowers",
    4: "Healthy Borrowers"
}
cluster_df['cluster_name'] = cluster_df['cluster'].map(cluster_names)


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_revol_util",
    text="avg_revol_util",
    title="Cluster-wise Average Revolving Utilization (%)",
    labels={"cluster_name": "Borrower Segment", "avg_revol_util": "Avg Revolving Util (%)"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg Revolving Util (%)", height=500)
fig.show()


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_dti",
    text="avg_dti",
    title="Cluster-wise Average Debt-to-Income Ratio",
    labels={"cluster_name": "Borrower Segment", "avg_dti": "Avg DTI"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg DTI", height=500)
fig.show()


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_total_bal_ex_mort",
    text="avg_total_bal_ex_mort",
    title="Cluster-wise Average Total Balance (Excl. Mortgage)",
    labels={"cluster_name": "Borrower Segment", "avg_total_bal_ex_mort": "Avg Total Balance"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg Total Balance", height=500)
fig.show()


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_num_120_pd",
    text="avg_num_120_pd",
    title="Cluster-wise Average Accounts 120+ Days Past Due",
    labels={"cluster_name": "Borrower Segment", "avg_num_120_pd": "Avg Accounts 120+ Days Past Due"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg Accounts 120+ Days Past Due", height=500)
fig.show()


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_delinq_2yrs",
    text="avg_delinq_2yrs",
    title="Cluster-wise Average Delinquencies (Last 2 Yrs)",
    labels={"cluster_name": "Borrower Segment", "avg_delinq_2yrs": "Avg Delinquencies"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg Delinquencies", height=500)
fig.show()


In [0]:
fig = px.bar(
    cluster_df,
    x="cluster_name",
    y="avg_annual_inc",
    text="avg_annual_inc",
    title="Cluster-wise Average Annual Income",
    labels={"cluster_name": "Borrower Segment", "avg_annual_inc": "Avg Annual Income ($)"},
    color="cluster_name",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(template="plotly_white", yaxis_title="Avg Annual Income ($)", height=500)
fig.show()


In [0]:

df_clusters = df_clusters.withColumn(
    "earliest_cr_line_dt",
    F.to_date(F.concat(F.lit("01-"), F.col("earliest_cr_line")), "dd-MMM-yyyy")
)

windowSpec = Window.orderBy("earliest_cr_line_dt").rowsBetween(-5, 0)

df_clusters = df_clusters.withColumn(
    "rolling_avg_delinq",
    F.avg("delinq_2yrs").over(windowSpec)
)

default_trends = df_clusters.groupBy(
    year("earliest_cr_line_dt").alias("year"),
    month("earliest_cr_line_dt").alias("month")
).agg(
    F.avg("rolling_avg_delinq").alias("rolling_avg_delinquencies")
).orderBy("year", "month")

default_trends.show(10)


In [0]:
default_trends_df = default_trends.toPandas()
default_trends_df = default_trends_df.dropna(subset=['year', 'month'])
default_trends_df['year_month'] = pd.to_datetime(
    default_trends_df['year'].astype(int).astype(str) + '-' +
    default_trends_df['month'].astype(int).astype(str) + '-01'
)

In [0]:


fig = px.line(
    default_trends_df,
    x='year_month',
    y='rolling_avg_delinquencies',
    title="Rolling Average Delinquencies by Account Opening Date",
    labels={
        'year_month': 'Account Opening Date', 
        'rolling_avg_delinquencies': 'Rolling Average Delinquencies (last 6 months)'
    },
    markers=True
)

fig.update_layout(
    xaxis_title='Account Opening Date',
    yaxis_title='Rolling Average Delinquencies (last 6 months)',
    template='plotly_white',
    height=600
)

fig.show()


# Geographic Risk Assessment: Average DTI by State

This bar chart highlights the **systemic financial health and market risk** across different U.S. states by analyzing the **average Debt-to-Income Ratio (DTI)**. DTI is a critical measure of the debt burden relative to income, giving insight into a region's overall borrower risk.

---

## 1. States with Highest Systemic Risk

The top of the chart shows states with the **highest average DTI**, indicating greater financial strain among borrowers:

- **Highest Risk States:** Utah (UT), Indiana (IN), Kentucky (KY) – average DTI approaches **24**.  
- **Inference:** Borrowers in these states dedicate a larger portion of their income to debt repayment. This signals higher **systemic risk** for lenders, as any economic downturn could quickly impact repayment ability.

---

## 2. States with Lowest Systemic Risk

The bottom of the chart shows states with the **lowest average DTI**, reflecting stronger financial resilience:

- **Lowest Risk States:** District of Columbia (DC), Massachusetts (MA), California (CA) – average DTI around **17–18**.  
- **Inference:** Borrowers carry a lighter debt load relative to income, suggesting **lower risk** and more stable markets, making these states attractive for lending.

---

## 3. Actionable Insights

- **Differentiated Pricing:** The 6–7 percentage point gap between high and low DTI states supports **state-specific underwriting** and **risk-based pricing**. Loans in high-DTI states may need **higher interest rates** or **stricter eligibility criteria**.  
- **Market Strategy:** Focus marketing and capital allocation on states with lower average DTI, as these are **lower-risk markets** with stronger potential for expansion.


In [0]:
from pyspark.sql import functions as F

# Compute average DTI, income, delinquencies, and total loans by state
state_stats = df_clusters.groupBy("addr_state").agg(
    F.avg("dti").alias("avg_dti"),
    F.avg("annual_inc").alias("avg_income"),
    F.avg("acc_now_delinq").alias("avg_delinq"),
    F.sum("loan_amnt").alias("total_loans")
).orderBy("addr_state")

state_stats.show(10)


In [0]:
import pandas as pd

state_df = state_stats.toPandas()

state_df = state_df.sort_values("avg_income", ascending=False)


### Cluster Visualization: Revolving Utilization vs Debt-to-Income Ratio

This scatter plot shows how our K-Means borrower segments are distributed when we look only at **Revolving Utilization (`revol_util`)** and **Debt-to-Income Ratio (`dti`)**. Each point represents a borrower, colored according to their cluster.  

### 1. Most Borrowers Are Packed Together
The bulk of borrowers across all clusters are squeezed into the lower-left corner of the plot (`dti < 5` and `revol_util < 2`). In this area, clusters overlap heavily, meaning that just looking at these two features doesn’t really separate them.  

### 2. These Features Don’t Explain the Clusters
Because most points are overlapping, it’s clear that `dti` and `revol_util` **aren’t the main drivers** behind the clusters. The real separation probably comes from other factors like income, past delinquency, loan grade, or home ownership.  

### 3. Outliers Stand Out
There are a few points with very high `dti` (up to ~45), mostly in the Affluent High-Debt or high-risk groups. While these extreme values pull the cluster averages slightly, they don’t represent the majority of borrowers.  

### 4. Centroids Don’t Help Much
The cluster centroids (marked as `X` on the plot) all sit very close together in the low `dti` and low `revol_util` area. This confirms that these two features alone aren’t enough to tell the clusters apart.  

### Takeaways

- If we tried to classify borrowers using only `dti` and revolving utilization, we would **fail to separate high-risk borrowers from prime borrowers**.  
- Understanding borrower risk requires looking at **multiple features** together, like income, past repayment behavior, and categorical loan attributes.  
- To see the actual separation between clusters, we need to rely on **PCA plots** or the **cluster profiles table**.  

In [0]:
cluster_df = cluster_stats.toPandas()

cluster_names = {
    0: "Prime Borrowers",
    1: "High-Risk Strained Borrowers",
    2: "Chronic High-Risk Borrowers",
    3: "Low-Debt Conservative Borrowers",
    4: "Affluent High-Debt Borrowers"
}

cluster_df['cluster_name'] = cluster_df['cluster'].map(cluster_names)


In [0]:
# ------------------------------
# 1. Sample & prepare data
# ------------------------------
sample_df = df_clusters.sample(fraction=0.02, seed=42).toPandas()

cluster_names = {
    0: "Prime Borrowers",
    1: "High-Risk Strained Borrowers",
    2: "Chronic High-Risk Borrowers",
    3: "Low-Debt Conservative Borrowers",
    4: "Affluent High-Debt Borrowers"
}
sample_df['cluster_name'] = sample_df['cluster'].map(cluster_names)

# ------------------------------
# 2. Scatter plot with jitter-like effect
# ------------------------------
import numpy as np
np.random.seed(42)
sample_df['revol_util_jitter'] = sample_df['revol_util'] + np.random.uniform(-1, 1, len(sample_df))
sample_df['dti_jitter'] = sample_df['dti'] + np.random.uniform(-0.5, 0.5, len(sample_df))

fig = px.scatter(
    sample_df,
    x='revol_util_jitter',
    y='dti_jitter',
    color='cluster_name',
    hover_data=['revol_util', 'dti'],  # show original values on hover
    title='K-Means Clusters: Revolving Utilization vs DTI (Interactive)',
    labels={'revol_util_jitter': 'Revolving Utilization', 'dti_jitter': 'Debt-to-Income Ratio'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    opacity=0.7,
    width=900,
    height=600
)

fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color='black')))
fig.show()


### 1. Feature Definition and Model Input

For this analysis, we trained a **K-Means clustering model** using **12 features**—a mix of numeric and categorical attributes—to capture a complete picture of borrower behavior and risk.

### Numeric Features

These numeric features were particularly useful for profiling the borrowers:

| Feature Name              | What it tells us                                      |
|----------------------------|-----------------------------------------------------|
| `annual_inc`               | Borrower’s gross annual income, indicating repayment capacity. |
| `dti`                      | Debt-to-income ratio, measuring financial strain.  |
| `revol_util`               | Revolving line utilization rate, reflecting credit reliance. |
| `total_bal_ex_mort`        | Non-housing debt load.                              |
| `delinq_2yrs`              | Number of delinquencies in the last 2 years, indicating recent credit stability. |
| `num_accts_ever_120_pd`    | History of severe credit issues.                   |

### Categorical Features

We one-hot encoded the following categorical variables to account for borrower and loan structure differences:

- `home_ownership`
- `purpose`
- `grade`
- `sub_grade`
- `term`
- `verification_status`

---

### 2. Cluster Validation and Visualization

To validate the clusters, we plotted a **PCA 2D projection** of the dataset.  

- The clusters appear **well-separated**, suggesting that K-Means did a good job segmenting borrowers despite the high-dimensional data.  
- **Cluster “Affluent High-Debt Borrowers”** stands out as isolated in the plot, highlighting a distinct borrower group.  
- Note that PCA1 and PCA2 together explain only about **9.4% of the variance**, meaning the separation relies on subtle interactions between multiple features rather than a single dominant variable.

---

### 3. Understanding the Borrower Segments

Analyzing the average feature values for each cluster helps us assign **descriptive profiles**:

| **Cluster Name**                   | **Key Characteristics**                                                                 | **Business Insight**                                                           |
|------------------------------------|----------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| **Prime Borrowers**                 | High income (82.1k USD), very low past delinquencies (0.46)                           | Most stable borrowers, ideal for offering the lowest rates.                   |
| **High-Risk Strained Borrowers**    | High debt load, highest `dti` (21.32), high `revol_util` (55.47%), past delinquencies 0.36 | Borrowers under financial strain; need close monitoring and risk-based pricing. |
| **Chronic High-Risk Borrowers**     | Poor past credit (0.76 accounts >120 days past due), low income (~70.5k USD)           | Likely candidates for loan denial; very high default risk.                    |
| **Low-Debt Conservative Borrowers** | Low debt load (`dti` 17.41, `revol_util` 33.44%)                                        | Low-risk segment; attractive target for competitive loan offers.              |
| **Affluent High-Debt Borrowers**    | High income (82.1k USD) but heavy non-mortgage debt (54.1k USD)                        | High borrowing capacity but risk must be managed carefully; isolated cluster. |

---

### 4. Risk Tiers

From the clusters, we can define **risk tiers** to guide lending decisions:

- **Low Risk:** Prime Borrowers – stable income and excellent repayment history.  
- **Medium Risk:** Affluent High-Debt Borrowers – high earning potential but existing debt requires caution.  
- **High Risk:** High-Risk Strained Borrowers – high debt and some delinquency; monitor closely.  
- **Very High Risk:** Chronic High-Risk Borrowers – poor credit history, low income; loans should be highly scrutinized.  
- **Low to Medium Risk:** Low-Debt Conservative Borrowers – financially responsible; low risk but less profitable than Prime Borrowers.

---

### 5. Takeaways

This clustering provides actionable insights:

- **Prime Borrowers** are the safest segment and can be offered **favorable loan terms**.  
- **High-Risk Strained Borrowers** require **higher caution and risk-adjusted pricing**.  
- **Chronic High-Risk Borrowers** represent the **highest default risk**; loans may need to be denied or heavily priced.  
- **Low-Debt Conservative Borrowers** are reliable, suitable for competitive rates.  
- **Affluent High-Debt Borrowers** have capacity but require **careful debt assessment**.  

Overall, these clusters allow lenders to adopt **tailored lending strategies** that balance risk and profitability effectively.

In [0]:
# ------------------------------
# 5. Feature Contributions to PCA Components
# ------------------------------
pc_loadings = pd.DataFrame(pca.components_, columns=all_feature_names, index=['PC1','PC2'])
print("Feature contributions to PCA components:")
display(pc_loadings.T.sort_values(by='PC1', ascending=False))


In [0]:
# ------------------------------
# 10. Stop Spark
# ------------------------------
# spark.stop()